# CASCADES v9 — Full Training Pipeline

Runs the complete CASCADES continual learning pipeline on an A100 GPU.

**What this does:**
1. Loads Qwen3-4B Heretic in 4-bit NF4
2. Injects CASCADES adapters (Stiefel manifold + Liquid Core)
3. Trains on 3 sequential reasoning tasks (495 examples)
4. Evaluates backward transfer + generative exact match

**Runtime:** ~5-10 min on A100, ~15 min on T4

## 1. Setup

In [ ]:
!git clone https://github.com/Bender1011001/CASCADES--continual-PEFT-for-Local-LLMs.git
%cd CASCADES--continual-PEFT-for-Local-LLMs
!pip install -q -r requirements.txt

## 2. Verify GPU

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("No GPU! Go to Runtime > Change runtime type > A100 GPU")

## 3. Run Full Training

This trains all 3 tasks with the fixed optimizer and runs generative evaluation.

In [ ]:
OUTPUT_PREFIX = "cascades_v9_colab"

!python train.py --eval_em --output_prefix {OUTPUT_PREFIX}

## 4. Results

In [ ]:
import pandas as pd

results_csv = f"{OUTPUT_PREFIX}_results.csv"
df = pd.read_csv(results_csv, index_col=0)

print("Accuracy Matrix (Proxy ACC %):")
print((df * 100).round(2).to_string())

# Backward Transfer
import numpy as np
num_tasks = len(df)
bwt_vals = [df.iloc[-1, i] - df.iloc[i, i] for i in range(num_tasks - 1)]
bwt = np.mean(bwt_vals) * 100
avg_acc = df.iloc[-1].mean() * 100

print(f"\nAverage Accuracy: {avg_acc:.2f}%")
print(f"Backward Transfer: {bwt:+.2f}%")
print(f"\nVerdict: {'Zero forgetting confirmed' if bwt >= 0 else 'Some forgetting detected'}")

## 5. Download Weights

Download the trained adapter weights to use locally.

In [ ]:
from google.colab import files
files.download(f"{OUTPUT_PREFIX}_weights.pt")
files.download(f"{OUTPUT_PREFIX}_results.csv")